# CNN Spatio-Temporal Stream — Deepfake Detection

## Two-Stream Late Fusion Architecture (Stream 2 of 2)

This notebook implements a **research-grade Spatio-Temporal CNN** using EfficientNet-B4 backbone with **BiLSTM Temporal Aggregation** for deepfake detection.

### Key Research Contributions:

1. **Temporal Modeling (BiLSTM + Multi-Head Attention)**
   - Unlike naive frame averaging, we model inter-frame dependencies
   - Detects temporal flickering, blending shifts, and motion anomalies
   - Enables detection of GAN/Diffusion artifacts that manifest across frames

2. **Grad-CAM Visualization**
   - Provides visual proof of what the model learns
   - Shows attention on facial regions (jawline, eyes, blending boundaries)
   - Essential for research paper methodology section

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                    SPATIO-TEMPORAL ARCHITECTURE                              │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   Video (T frames) ──→ MTCNN Face Detection ──→ T × (224, 224, 3) crops     │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  SPATIAL FEATURE EXTRACTION                                          │   │
│   │  EfficientNet-B4 (pretrained, 1792-dim features per frame)          │   │
│   │                                                                      │   │
│   │      Frame 1 ──→ [f₁]                                               │   │
│   │      Frame 2 ──→ [f₂]     ──→ Feature Sequence (T × 1792)           │   │
│   │      ...                                                             │   │
│   │      Frame T ──→ [fₜ]                                               │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  TEMPORAL AGGREGATION (BiLSTM + Multi-Head Attention)               │   │
│   │                                                                      │   │
│   │  [f₁, f₂, ..., fₜ] ──→ BiLSTM (2-layer, bidirectional)              │   │
│   │                                ↓                                     │   │
│   │                    Multi-Head Self-Attention (4 heads)               │   │
│   │                                ↓                                     │   │
│   │                    Weighted Temporal Pooling                         │   │
│   │                                ↓                                     │   │
│   │                    Video-Level Representation (512-dim)              │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  CLASSIFIER                                                          │   │
│   │  Linear(512 → 256) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(256 → 128) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(128 → 1) → Sigmoid                                          │   │
│   │                        ↓                                             │   │
│   │                    P_CNN (0 = Real, 1 = Fake)                        │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘

                    ↓ LATE FUSION ↓

        P_final = w₁ × P_CNN + w₂ × P_rPPG (from Stream 1)
```

**Output:** `cnn_predictions.csv` with video-level P_CNN scores for Late Fusion

## 1. Setup & Imports

In [ ]:
!pip install "numpy<2" --force-reinstall

In [ ]:
!pip install timm==0.9.16 transformers==4.38.2 --force-reinstall

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

def install_packages():
    packages = [
        "facenet-pytorch",
        "timm",
        "albumentations",
        "opencv-python-headless",
    ]
    for pkg in packages:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    print("✓ All packages installed")

install_packages()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS (P100 COMPATIBLE - STRICT FP32, NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════
# FIX: Added worker_init_fn for DataLoader numpy compatibility
# ═══════════════════════════════════════════════════════════════════════════════

import os
import re
import gc
import cv2
import json
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
from tqdm.auto import tqdm
import io
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ═══════════════════════════════════════════════════════════════════════════════
# P100 CRITICAL: NO AMP/MIXED PRECISION
# ═══════════════════════════════════════════════════════════════════════════════

import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, roc_curve, precision_score, recall_score

from scipy.optimize import brentq
from scipy.interpolate import interp1d

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY + WORKER INIT (FIX FOR NUMPY ERROR)
# ═══════════════════════════════════════════════════════════════════════════════

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# CRITICAL FIX: Worker init function for DataLoader
# This ensures numpy is available in worker processes
def worker_init_fn(worker_id):
    """Initialize worker with numpy and proper seeding."""
    import numpy as np
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Generator for reproducible DataLoader
g = torch.Generator()
g.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {cap[0]}.{cap[1]}")
    if cap[0] < 7:
        print(f"\n⚠️ PASCAL GPU DETECTED - Using STRICT FP32 (no AMP)")
        print(f"✓ Worker init function configured for DataLoader")

print("\n✓ All imports successful")
print("✓ Worker init function defined for numpy compatibility")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION (Research Paper Ready - P100 Optimized)
# ═══════════════════════════════════════════════════════════════════════════════
#
# CRITICAL P100 MEMORY OPTIMIZATION:
#   - BATCH_SIZE = 4 (reduced from 8 to prevent OOM)
#   - GRAD_ACCUMULATION_STEPS = 2 (effective batch size = 8)
#   - Each step: 4 videos × 15 frames = 60 images
#   - Accumulated: 8 videos × 15 frames = 120 images effective batch
#
# ═══════════════════════════════════════════════════════════════════════════════

import math
import re

class Config:
    # Experiment tracking (for reproducibility)
    EXPERIMENT_NAME = "CNN_SpatioTemporal_BiLSTM_Attn_FocalLoss"
    EXPERIMENT_VERSION = "v3.0_research"
    
    # Dataset paths (Kaggle)
    DATASET_ROOT = "/kaggle/input/datasets/lalith023/400videos/content/drive/MyDrive/face_dataset_dip"
    REAL_DIR = os.path.join(DATASET_ROOT, "real_videos")
    FAKE_DIR = os.path.join(DATASET_ROOT, "deepfake_videos")
    OUTPUT_DIR = "/kaggle/working"

    # Frame extraction
    FRAMES_PER_VIDEO = 15
    IMG_SIZE = 224

    # ═══════════════════════════════════════════════════════════════════════════
    # CRITICAL: P100 Memory Optimization with Gradient Accumulation
    # ═══════════════════════════════════════════════════════════════════════════
    # 
    # Problem: batch_size=8 × 15 frames = 120 images → OOM on P100
    # Solution: batch_size=4 × grad_accum=2 → effective batch of 8
    #
    BATCH_SIZE = 4  # REDUCED: 4 videos × 15 frames = 60 images per step
    GRAD_ACCUMULATION_STEPS = 2  # Effective batch size = 4 × 2 = 8 videos
    
    NUM_EPOCHS = 20
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    NUM_WORKERS = 2
    WARMUP_RATIO = 0.1  # 10% warmup steps
    
    # Focal Loss parameters (for hard example mining)
    FOCAL_ALPHA = 0.25  # Weight for positive class
    FOCAL_GAMMA = 2.0   # Focusing parameter (higher = more focus on hard examples)

    # Model - Backbone
    MODEL_NAME = "efficientnet_b4"
    DROPOUT = 0.4
    HIDDEN_DIM = 256
    USE_GRADIENT_CHECKPOINTING = True  # Saves ~40% VRAM on P100

    # Model - Temporal Aggregator (BiLSTM for inter-frame temporal modeling)
    TEMPORAL_TYPE = "bilstm_attention"  # Options: "bilstm", "bilstm_attention", "transformer"
    LSTM_HIDDEN = 256
    LSTM_LAYERS = 2
    ATTENTION_HEADS = 4
    FREEZE_BACKBONE = False  # Set True for faster training, False for best accuracy

    # K-Fold Cross-Validation (RESEARCH REQUIREMENT)
    K_FOLDS = 5
    CURRENT_FOLD = 0  # Which fold to run (0-4), or set to -1 for single split
    
    # Identity-based splitting (CRITICAL: prevents data leakage)
    USE_IDENTITY_SPLIT = True  # Group by person identity, not random
    
    # Split (used when CURRENT_FOLD = -1)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.2
    
    @classmethod
    def to_dict(cls):
        """Export config for logging and reproducibility."""
        return {k: v for k, v in vars(cls).items() 
                if not k.startswith('_') and not callable(v)}

cfg = Config()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Save config for reproducibility
import json
config_path = os.path.join(cfg.OUTPUT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(cfg.to_dict(), f, indent=2, default=str)

# Calculate effective batch size
effective_batch_size = cfg.BATCH_SIZE * cfg.GRAD_ACCUMULATION_STEPS

print("="*70)
print(f"CNN SPATIO-TEMPORAL STREAM: {cfg.EXPERIMENT_NAME}")
print(f"Version: {cfg.EXPERIMENT_VERSION}")
print("="*70)
print(f"\n📊 DATASET:")
print(f"  Dataset root: {cfg.DATASET_ROOT}")
print(f"  Frames per video: {cfg.FRAMES_PER_VIDEO}")
print(f"  Image size: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")

print(f"\n⚡ P100 MEMORY OPTIMIZATION:")
print(f"  Physical batch size: {cfg.BATCH_SIZE} videos")
print(f"  Gradient accumulation: {cfg.GRAD_ACCUMULATION_STEPS} steps")
print(f"  Effective batch size: {effective_batch_size} videos")
print(f"  Images per physical batch: {cfg.BATCH_SIZE * cfg.FRAMES_PER_VIDEO}")
print(f"  Gradient checkpointing: {cfg.USE_GRADIENT_CHECKPOINTING}")

print(f"\n🔧 TRAINING:")
print(f"  Epochs: {cfg.NUM_EPOCHS}")
print(f"  Learning rate: {cfg.LEARNING_RATE} (with {cfg.WARMUP_RATIO*100:.0f}% warmup)")
print(f"  Loss: Focal Loss (α={cfg.FOCAL_ALPHA}, γ={cfg.FOCAL_GAMMA})")

print(f"\n🧠 MODEL:")
print(f"  Backbone: {cfg.MODEL_NAME}")
print(f"  Temporal aggregator: {cfg.TEMPORAL_TYPE}")
print(f"  LSTM hidden: {cfg.LSTM_HIDDEN}, layers: {cfg.LSTM_LAYERS}")

print(f"\n📈 EVALUATION:")
print(f"  K-Fold CV: {cfg.K_FOLDS} folds (current fold: {cfg.CURRENT_FOLD})")
print(f"  Identity-based split: {cfg.USE_IDENTITY_SPLIT}")
print(f"  Config saved to: {config_path}")
print("="*70)


## 2. Frame Extraction & Face Detection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MTCNN FACE DETECTOR
# ═══════════════════════════════════════════════════════════════════════════════

class FaceExtractor:
    """
    Extracts faces from video frames using MTCNN.
    Falls back to center crop if no face is detected.
    """
    
    def __init__(self, device, img_size=224, margin=40):
        self.device = device
        self.img_size = img_size
        self.margin = margin
        
        # Initialize MTCNN with optimized settings
        self.mtcnn = MTCNN(
            image_size=img_size,
            margin=margin,
            min_face_size=60,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=False,  # Raw pixel values [0, 255], no [-1, 1] normalization
            device=device,
            keep_all=False,  # Only largest face
        )
        print(f"✓ MTCNN initialized on {device}")
    
    def extract_face(self, frame: np.ndarray) -> Optional[np.ndarray]:
        """
        Extract face from a single frame.
        Returns: Face crop as numpy array (H, W, 3) or None if failed.
        """
        try:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # Detect face
            face = self.mtcnn(pil_img)
            
            if face is not None:
                # MTCNN returns tensor (C, H, W), convert to numpy (H, W, C)
                # Since post_process=False, values are already 0-255
                face_np = face.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                return face_np
            else:
                # Fallback: center crop
                return self._center_crop(frame_rgb)
                
        except Exception as e:
            # Fallback: center crop
            return self._center_crop(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    def _center_crop(self, frame: np.ndarray) -> np.ndarray:
        """Center crop fallback when face detection fails."""
        h, w = frame.shape[:2]
        size = min(h, w)
        y = (h - size) // 2
        x = (w - size) // 2
        crop = frame[y:y+size, x:x+size]
        return cv2.resize(crop, (self.img_size, self.img_size))


# Initialize face extractor
face_extractor = FaceExtractor(DEVICE, img_size=cfg.IMG_SIZE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO FRAME EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_frames_from_video(video_path: str, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract n evenly spaced frames from a video.
    
    Args:
        video_path: Path to video file
        n_frames: Number of frames to extract
        
    Returns:
        List of frame arrays (BGR format)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return []  # Corrupted video
    
    if total_frames < n_frames:
        # If video has fewer frames, extract all
        frame_indices = list(range(total_frames))
    else:
        # Evenly spaced frame indices
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    
    cap.release()
    return frames


def process_video(video_path: str, face_extractor: FaceExtractor, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract frames and detect faces from a video.
    
    Returns:
        List of face crops as numpy arrays
    """
    frames = extract_frames_from_video(video_path, n_frames)
    
    face_crops = []
    for frame in frames:
        face = face_extractor.extract_face(frame)
        if face is not None:
            face_crops.append(face)
    
    return face_crops


print("✓ Frame extraction functions defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROCESS ALL VIDEOS
# ═══════════════════════════════════════════════════════════════════════════════

def collect_video_paths():
    """Collect all video paths with labels."""
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv')
    
    videos = []
    
    print("\n" + "="*70)
    print("DATASET PATH VALIDATION")
    print("="*70)
    print(f"  DATASET_ROOT: {cfg.DATASET_ROOT}")
    print(f"  REAL_DIR:     {cfg.REAL_DIR}")
    print(f"  FAKE_DIR:     {cfg.FAKE_DIR}")
    
    # Check if paths exist
    print(f"\n  DATASET_ROOT exists: {os.path.exists(cfg.DATASET_ROOT)}")
    print(f"  REAL_DIR exists:     {os.path.exists(cfg.REAL_DIR)}")
    print(f"  FAKE_DIR exists:     {os.path.exists(cfg.FAKE_DIR)}")
    
    # If dataset root doesn't exist, try to find it
    if not os.path.exists(cfg.DATASET_ROOT):
        print("\n⚠️  DATASET_ROOT not found! Searching for alternatives...")
        
        # Common Kaggle input paths to check
        search_paths = [
            "/kaggle/input",
        ]
        
        for search_path in search_paths:
            if os.path.exists(search_path):
                print(f"\n  Contents of {search_path}:")
                try:
                    for item in os.listdir(search_path)[:10]:
                        item_path = os.path.join(search_path, item)
                        print(f"    - {item} {'(dir)' if os.path.isdir(item_path) else '(file)'}")
                except Exception as e:
                    print(f"    Error listing: {e}")
        
        raise FileNotFoundError(
            f"Dataset root not found at: {cfg.DATASET_ROOT}\n"
            f"Please update cfg.DATASET_ROOT in the Config cell to the correct path."
        )
    
    # Real videos (label = 0)
    real_count = 0
    if os.path.exists(cfg.REAL_DIR):
        for f in sorted(os.listdir(cfg.REAL_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.REAL_DIR, f),
                    'label': 0  # Real
                })
                real_count += 1
        print(f"\n✓ Found {real_count} real videos in: {cfg.REAL_DIR}")
    else:
        print(f"\n⚠️  REAL_DIR not found: {cfg.REAL_DIR}")
        # List contents of DATASET_ROOT to help debug
        print(f"  Contents of DATASET_ROOT ({cfg.DATASET_ROOT}):")
        for item in os.listdir(cfg.DATASET_ROOT):
            print(f"    - {item}")
    
    # Fake videos (label = 1)
    fake_count = 0
    if os.path.exists(cfg.FAKE_DIR):
        for f in sorted(os.listdir(cfg.FAKE_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.FAKE_DIR, f),
                    'label': 1  # Fake
                })
                fake_count += 1
        print(f"✓ Found {fake_count} fake videos in: {cfg.FAKE_DIR}")
    else:
        print(f"\n⚠️  FAKE_DIR not found: {cfg.FAKE_DIR}")
        # List contents of DATASET_ROOT to help debug
        if os.path.exists(cfg.DATASET_ROOT):
            print(f"  Contents of DATASET_ROOT ({cfg.DATASET_ROOT}):")
            for item in os.listdir(cfg.DATASET_ROOT):
                print(f"    - {item}")
    
    if len(videos) == 0:
        raise ValueError(
            f"No videos found!\n"
            f"  REAL_DIR: {cfg.REAL_DIR} (exists: {os.path.exists(cfg.REAL_DIR)})\n"
            f"  FAKE_DIR: {cfg.FAKE_DIR} (exists: {os.path.exists(cfg.FAKE_DIR)})\n"
            f"Please check that your video directories contain .mp4/.avi/.mov/.mkv files."
        )
    
    print("="*70)
    return videos

# Collect videos
all_videos = collect_video_paths()
print(f"\nTotal videos found: {len(all_videos)}")
print(f"  Real: {sum(1 for v in all_videos if v['label'] == 0)}")
print(f"  Fake: {sum(1 for v in all_videos if v['label'] == 1)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTRACT ALL FACES (with caching)
# ═══════════════════════════════════════════════════════════════════════════════

def extract_all_faces(videos: List[dict], face_extractor: FaceExtractor) -> Dict[str, List[np.ndarray]]:
    """
    Extract faces from all videos.
    
    Returns:
        Dictionary mapping video_id to list of face crops
    """
    face_data = {}
    
    print("\n" + "="*70)
    print("EXTRACTING FACES FROM VIDEOS")
    print("="*70)
    
    failed_videos = []
    
    for video in tqdm(videos, desc="Processing videos"):
        video_id = video['video_id']
        video_path = video['path']
        
        faces = process_video(video_path, face_extractor, cfg.FRAMES_PER_VIDEO)
        
        if len(faces) >= 3:  # Require at least 3 valid faces
            face_data[video_id] = faces
        else:
            failed_videos.append(video_id)
    
    print(f"\n✓ Successfully processed: {len(face_data)} videos")
    print(f"✗ Failed/insufficient faces: {len(failed_videos)} videos")
    
    # Memory cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return face_data

# Extract faces
face_data = extract_all_faces(all_videos, face_extractor)

# DO NOT shrink all_videos — this would misalign train_test_split indices
# with the ML notebook. Downstream code (Dataset, predict) already checks
# `if video_id in face_data` to safely skip missing videos.

# Validate we have face data
if len(face_data) == 0:
    raise ValueError("ERROR: No videos with valid face detections! Check face extraction.")
print(f"\nVideos with valid faces: {len(face_data)}/{len(all_videos)}")

## 3. Dataset Creation & Data Leakage Prevention

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IDENTITY-BASED DATA SPLITTING (CRITICAL: Prevents Data Leakage)
# ═══════════════════════════════════════════════════════════════════════════════
#
# PROBLEM: Random train/test split causes IDENTITY LEAKAGE:
#   - Real Person A in training, Fake Person A in validation
#   - Model memorizes faces, not deepfake artifacts
#   - Inflated metrics that don't generalize
#
# SOLUTION: Group videos by source identity BEFORE splitting:
#   - All videos of Person A go to train OR val, never both
#   - Forces model to learn manipulation artifacts, not faces
#   - Required by IEEE T-IFS, CVPR, and top security venues
#
# ═══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import StratifiedKFold, GroupKFold, StratifiedGroupKFold
from collections import defaultdict


def extract_identity_from_filename(filename: str) -> str:
    """
    Extract identity/person ID from video filename.
    
    Common deepfake dataset naming conventions:
    - FaceForensics++: {id}_{seq}.mp4 → identity = id
    - Celeb-DF: id{num}_{seq}.mp4 → identity = id{num}
    - DFDC: {subject_id}_{video_id}.mp4 → identity = subject_id
    - Custom: video_{person}_{seq}.mp4 → identity = person
    
    Fallback: Use first segment before underscore/dash
    
    Args:
        filename: Video filename (with or without extension)
    
    Returns:
        String identifier for the source identity
    """
    # Remove extension
    name = os.path.splitext(os.path.basename(filename))[0]
    
    # Common patterns for identity extraction
    patterns = [
        # FaceForensics++: 123_456.mp4 → 123
        r'^(\d+)_\d+$',
        # Celeb-DF: id0_id1_0000.mp4 → id0_id1
        r'^(id\d+_id\d+)_\d+$',
        # DFDC style: abcdef_0.mp4 → abcdef
        r'^([a-zA-Z]+\d*)_\d+$',
        # General: person1_video2.mp4 → person1
        r'^([^_]+)_.*$',
    ]
    
    for pattern in patterns:
        match = re.match(pattern, name)
        if match:
            return match.group(1)
    
    # Fallback: first part before underscore or full name
    parts = re.split(r'[_\-]', name)
    return parts[0] if parts else name


def extract_identity_from_path(video_path: str, label: int) -> str:
    """
    Extract identity from video path, considering label.
    
    For deepfake datasets, the identity is typically embedded in the path:
    - /real_videos/person1/video.mp4 → person1_real
    - /fake_videos/person1_to_person2/video.mp4 → person1_fake
    
    We append label to prevent real/fake versions of same person
    from being split together (they should be, but this is safer).
    """
    filename = os.path.basename(video_path)
    identity = extract_identity_from_filename(filename)
    
    # Optionally include parent folder as part of identity
    parent = os.path.basename(os.path.dirname(video_path))
    if parent and parent not in ['real_videos', 'fake_videos', 'deepfake_videos']:
        identity = f"{parent}_{identity}"
    
    return identity


def assign_identities_to_videos(videos: List[dict]) -> List[dict]:
    """
    Assign identity labels to each video for group-based splitting.
    
    Args:
        videos: List of video dicts with 'video_id', 'label', 'path'
    
    Returns:
        Same list with 'identity' field added
    """
    for video in videos:
        video['identity'] = extract_identity_from_path(
            video.get('path', video['video_id']), 
            video['label']
        )
    
    # Print identity statistics
    identity_counts = defaultdict(int)
    for v in videos:
        identity_counts[v['identity']] += 1
    
    print(f"  Unique identities found: {len(identity_counts)}")
    print(f"  Videos per identity: min={min(identity_counts.values())}, "
          f"max={max(identity_counts.values())}, "
          f"mean={np.mean(list(identity_counts.values())):.1f}")
    
    return videos


def create_identity_aware_kfold_splits(videos: List[dict], n_splits: int = 5, seed: int = 42):
    """
    Create K-Fold splits that respect identity boundaries.
    
    CRITICAL FOR RESEARCH: Ensures no identity appears in both train and val.
    Uses StratifiedGroupKFold to maintain class balance while respecting groups.
    
    Args:
        videos: List of video dicts with 'identity' and 'label' fields
        n_splits: Number of folds
        seed: Random seed
    
    Returns:
        List of fold dicts with 'train' and 'val' video lists
    """
    # Assign identities if not already done
    if 'identity' not in videos[0]:
        videos = assign_identities_to_videos(videos)
    
    labels = np.array([v['label'] for v in videos])
    groups = np.array([v['identity'] for v in videos])
    indices = np.arange(len(videos))
    
    # Try StratifiedGroupKFold (maintains class balance + respects groups)
    try:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splitter = sgkf.split(indices, labels, groups)
    except ValueError:
        # Fallback to GroupKFold if stratification fails
        print("  Warning: Using GroupKFold (stratification not possible)")
        gkf = GroupKFold(n_splits=n_splits)
        splitter = gkf.split(indices, labels, groups)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(splitter):
        train_videos = [videos[i] for i in train_idx]
        val_videos = [videos[i] for i in val_idx]
        
        # Verify NO IDENTITY LEAKAGE
        train_identities = set(v['identity'] for v in train_videos)
        val_identities = set(v['identity'] for v in val_videos)
        leaked = train_identities & val_identities
        
        if leaked:
            print(f"  ⚠ WARNING: Identity leakage detected in fold {fold_idx}: {len(leaked)} identities")
        else:
            # Also verify video IDs don't overlap
            train_ids = set(v['video_id'] for v in train_videos)
            val_ids = set(v['video_id'] for v in val_videos)
            assert len(train_ids & val_ids) == 0, f"Video ID leakage in fold {fold_idx}!"
        
        folds.append({
            'fold': fold_idx,
            'train': train_videos,
            'val': val_videos,
            'train_real': sum(1 for v in train_videos if v['label'] == 0),
            'train_fake': sum(1 for v in train_videos if v['label'] == 1),
            'val_real': sum(1 for v in val_videos if v['label'] == 0),
            'val_fake': sum(1 for v in val_videos if v['label'] == 1),
            'train_identities': len(set(v['identity'] for v in train_videos)),
            'val_identities': len(set(v['identity'] for v in val_videos)),
        })
    
    return folds


def create_random_kfold_splits(videos: List[dict], n_splits: int = 5, seed: int = 42):
    """
    Create standard K-Fold splits (random, but video-level).
    
    Use only when identity information is not available.
    WARNING: May cause identity leakage!
    """
    labels = np.array([v['label'] for v in videos])
    indices = np.arange(len(videos))
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        train_videos = [videos[i] for i in train_idx]
        val_videos = [videos[i] for i in val_idx]
        
        folds.append({
            'fold': fold_idx,
            'train': train_videos,
            'val': val_videos,
            'train_real': sum(1 for v in train_videos if v['label'] == 0),
            'train_fake': sum(1 for v in train_videos if v['label'] == 1),
            'val_real': sum(1 for v in val_videos if v['label'] == 0),
            'val_fake': sum(1 for v in val_videos if v['label'] == 1),
        })
    
    return folds


# ═══════════════════════════════════════════════════════════════════════════════
# CREATE SPLITS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("DATA SPLIT CONFIGURATION")
print("="*70)

# Assign identities to all videos
print("\n📋 Extracting identities from video filenames...")
all_videos = assign_identities_to_videos(all_videos)

if cfg.USE_IDENTITY_SPLIT:
    print(f"\n🔒 IDENTITY-AWARE SPLITTING (Prevents Data Leakage)")
    print("-"*50)
    
    kfold_splits = create_identity_aware_kfold_splits(
        all_videos, n_splits=cfg.K_FOLDS, seed=SEED
    )
    
    # Print fold statistics
    print("\nFold Statistics (Identity-Aware):")
    for fold in kfold_splits:
        print(f"  Fold {fold['fold'] + 1}: "
              f"Train={len(fold['train'])} ({fold['train_identities']} identities), "
              f"Val={len(fold['val'])} ({fold['val_identities']} identities)")
else:
    print(f"\n⚠ RANDOM SPLITTING (May cause identity leakage!)")
    print("-"*50)
    
    kfold_splits = create_random_kfold_splits(
        all_videos, n_splits=cfg.K_FOLDS, seed=SEED
    )
    
    print("\nFold Statistics (Random):")
    for fold in kfold_splits:
        print(f"  Fold {fold['fold'] + 1}: Train={len(fold['train'])}, Val={len(fold['val'])}")

# Select current fold
if cfg.CURRENT_FOLD >= 0:
    current_split = kfold_splits[cfg.CURRENT_FOLD]
    train_videos = current_split['train']
    val_videos = current_split['val']
    
    print(f"\n✓ Using Fold {cfg.CURRENT_FOLD + 1}/{cfg.K_FOLDS}")
    print(f"  Train: {len(train_videos)} videos (Real: {current_split['train_real']}, Fake: {current_split['train_fake']})")
    print(f"  Val: {len(val_videos)} videos (Real: {current_split['val_real']}, Fake: {current_split['val_fake']})")
else:
    # Simple single split
    from sklearn.model_selection import train_test_split
    labels = [v['label'] for v in all_videos]
    train_videos, val_videos = train_test_split(
        all_videos, train_size=cfg.TRAIN_RATIO, random_state=SEED, stratify=labels
    )
    print(f"\n✓ Using single {cfg.TRAIN_RATIO*100:.0f}%-{cfg.VAL_RATIO*100:.0f}% split")

# Final verification - NO LEAKAGE
if cfg.USE_IDENTITY_SPLIT:
    train_ids = set(v['identity'] for v in train_videos)
    val_ids = set(v['identity'] for v in val_videos)
    assert len(train_ids & val_ids) == 0, "IDENTITY LEAKAGE DETECTED!"
    print(f"\n✅ NO IDENTITY LEAKAGE: {len(train_ids)} train identities, {len(val_ids)} val identities")

# Also verify video IDs
train_video_ids = set(v['video_id'] for v in train_videos)
val_video_ids = set(v['video_id'] for v in val_videos)
assert len(train_video_ids & val_video_ids) == 0, "VIDEO ID LEAKAGE DETECTED!"
print("✅ NO VIDEO LEAKAGE: Train and Val sets are completely separate")
print("="*70)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA AUGMENTATION + FOCAL LOSS + ACCURACY IMPROVEMENTS
# ═══════════════════════════════════════════════════════════════════════════════
# FIXES:
# 1. Custom ToTensor that handles numpy properly in workers
# 2. Focal Loss for class imbalance
# 3. Optimal threshold finder
# 4. Class weight calculation
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import cv2

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ─── CUSTOM TENSOR TRANSFORM (FIX FOR NUMPY WORKER ERROR) ────────────────────
class SafeToTensor(A.ImageOnlyTransform):
    """
    Safe ToTensor that works in DataLoader workers.
    Fixes: RuntimeError: Numpy is not available
    """
    def __init__(self, always_apply=True, p=1.0):
        super(SafeToTensor, self).__init__(always_apply, p)
    
    def apply(self, img, **params):
        # Ensure numpy is imported in this scope (fixes worker issue)
        import numpy as np
        
        if isinstance(img, np.ndarray):
            # HWC to CHW, then to tensor
            if img.ndim == 3:
                img = img.transpose(2, 0, 1)
            return torch.from_numpy(img.copy()).float()
        return img
    
    def get_transform_init_args_names(self):
        return ()


def get_train_transforms():
    """Training augmentations for deepfake detection."""
    return A.Compose([
        # Geometric (mild)
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, 
            scale_limit=0.05, 
            rotate_limit=10, 
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.3
        ),
        
        # Color augmentations
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),
        
        # Compression artifacts
        A.ImageCompression(quality_lower=50, quality_upper=100, p=0.5),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=0.3),
        
        # Noise and blur
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        
        # Occlusion
        A.CoarseDropout(max_holes=4, max_height=20, max_width=20, fill_value=0, p=0.2),
        
        # Normalize and convert (FIXED: use SafeToTensor)
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        SafeToTensor(),  # FIXED: Custom tensor conversion
    ])


def get_val_transforms():
    """Validation transforms: only normalization."""
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        SafeToTensor(),  # FIXED: Custom tensor conversion
    ])


# ─── FOCAL LOSS (CLASS IMBALANCE FIX) ────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance in deepfake detection.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        # Ensure inputs are probabilities
        if inputs.dim() > 1:
            # Multi-class: take the fake class probability
            p = torch.sigmoid(inputs.squeeze())
        else:
            p = torch.sigmoid(inputs)
        
        # Focal loss computation
        ce_loss = F.binary_cross_entropy_with_logits(inputs.squeeze(), targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        
        # Alpha weighting
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        loss = alpha_t * focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ─── OPTIMAL THRESHOLD FINDER ────────────────────────────────────────────────
def find_optimal_threshold(y_true, y_prob):
    """Find threshold that maximizes F1 score."""
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.2, 0.8, 0.02):
        preds = (y_prob >= thresh).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1


# ─── CLASS WEIGHT CALCULATOR ─────────────────────────────────────────────────
def calculate_class_weights(labels):
    """Calculate class weights for imbalanced data."""
    import numpy as np
    labels = np.array(labels)
    n_samples = len(labels)
    n_class_0 = (labels == 0).sum()
    n_class_1 = (labels == 1).sum()
    
    weight_0 = n_samples / (2 * n_class_0) if n_class_0 > 0 else 1.0
    weight_1 = n_samples / (2 * n_class_1) if n_class_1 > 0 else 1.0
    
    return {0: weight_0, 1: weight_1}


print("✓ Augmentation pipelines defined (FIXED)")
print("  - SafeToTensor: Worker-safe tensor conversion")
print("  - FocalLoss: Handles class imbalance")
print("  - find_optimal_threshold: Better F1 score")
print("  - calculate_class_weights: For weighted sampling")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PYTORCH DATASETS (FIXED FOR DATALOADER WORKERS)
# ═══════════════════════════════════════════════════════════════════════════════
# FIX: Added numpy import in __getitem__ for worker compatibility
# ═══════════════════════════════════════════════════════════════════════════════

class DeepfakeFaceDataset(Dataset):
    """Frame-level deepfake detection dataset."""

    def __init__(self, videos: List[dict], face_data: Dict[str, List[np.ndarray]],
                 transform=None, is_train=True):
        self.transform = transform
        self.is_train = is_train

        self.samples = []
        for video in videos:
            video_id = video['video_id']
            label = video['label']

            if video_id in face_data:
                for face in face_data[video_id]:
                    self.samples.append((face, label, video_id))

        print(f"  Frame-level dataset: {len(self.samples)} samples from {len(videos)} videos")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        import numpy as np  # CRITICAL: Import in worker scope
        
        face, label, video_id = self.samples[idx]
        
        # Ensure face is a proper numpy array
        face = np.asarray(face, dtype=np.uint8)

        if self.transform:
            augmented = self.transform(image=face)
            face = augmented['image']

        return {
            'image': face,
            'label': torch.tensor(label, dtype=torch.float32),
            'video_id': video_id
        }


class DeepfakeVideoDataset(Dataset):
    """
    Video-level deepfake detection with temporal modeling.
    Each sample is ALL frames from one video.
    """

    def __init__(self, videos: List[dict], face_data: Dict[str, List[np.ndarray]],
                 transform=None, max_frames: int = 15, is_train: bool = True):
        self.transform = transform
        self.max_frames = max_frames
        self.is_train = is_train

        self.videos = []
        for video in videos:
            video_id = video['video_id']
            if video_id in face_data and len(face_data[video_id]) >= 3:
                self.videos.append({
                    'video_id': video_id,
                    'label': video['label'],
                    'faces': face_data[video_id]
                })

        print(f"  Video-level dataset: {len(self.videos)} videos")

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        import numpy as np  # CRITICAL: Import in worker scope
        
        video = self.videos[idx]
        faces = video['faces']
        label = video['label']
        video_id = video['video_id']

        # Sample or pad frames
        if len(faces) >= self.max_frames:
            indices = np.linspace(0, len(faces) - 1, self.max_frames, dtype=int)
            selected_faces = [faces[i] for i in indices]
            mask = torch.ones(self.max_frames, dtype=torch.bool)
        else:
            selected_faces = list(faces)
            while len(selected_faces) < self.max_frames:
                selected_faces.append(faces[-1])
            mask = torch.zeros(self.max_frames, dtype=torch.bool)
            mask[:len(faces)] = True

        # Apply transforms
        frame_tensors = []
        for face in selected_faces:
            face = np.asarray(face, dtype=np.uint8)  # Ensure proper numpy array
            if self.transform:
                augmented = self.transform(image=face)
                frame_tensor = augmented['image']
            else:
                frame_tensor = torch.from_numpy(face.transpose(2, 0, 1).copy()).float() / 255.0
            frame_tensors.append(frame_tensor)

        frames = torch.stack(frame_tensors, dim=0)

        return {
            'frames': frames,
            'label': torch.tensor(label, dtype=torch.float32),
            'video_id': video_id,
            'mask': mask
        }


# ─── CREATE DATASETS ─────────────────────────────────────────────────────────
print("\nCreating VIDEO-LEVEL datasets...")
train_dataset = DeepfakeVideoDataset(
    train_videos, face_data, transform=get_train_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO, is_train=True
)
val_dataset = DeepfakeVideoDataset(
    val_videos, face_data, transform=get_val_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO, is_train=False
)

# Calculate class weights
train_labels = [v['label'] for v in train_videos]
class_weights = calculate_class_weights(train_labels)
print(f"\nClass weights: {class_weights}")

# ─── CREATE DATALOADERS (FIXED) ──────────────────────────────────────────────
# CRITICAL FIX: Use num_workers=2 with worker_init_fn
# If still failing, set num_workers=0

NUM_WORKERS = 2  # Reduced from 4 for stability

train_loader = DataLoader(
    train_dataset, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=True,
    num_workers=NUM_WORKERS, 
    pin_memory=True, 
    drop_last=True,
    worker_init_fn=worker_init_fn,  # CRITICAL FIX
    generator=g,
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=False,
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    worker_init_fn=worker_init_fn,  # CRITICAL FIX
)

print(f"\n✓ Train loader: {len(train_loader)} batches")
print(f"✓ Val loader: {len(val_loader)} batches")
print(f"✓ Workers: {NUM_WORKERS} (with worker_init_fn)")
print(f"✓ Each batch: {cfg.BATCH_SIZE} videos × {cfg.FRAMES_PER_VIDEO} frames")

## 4. CNN Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIO-TEMPORAL CNN MODEL (P100 COMPATIBLE - STRICT FP32)
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalAttention(nn.Module):
    def __init__(self, feature_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        key_padding_mask = ~mask if mask is not None else None
        attn_out, attn_weights = self.attention(x, x, x, key_padding_mask=key_padding_mask)
        x = self.layer_norm(x + self.dropout(attn_out))
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).float()
            x = x * mask_expanded
            pooled = x.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            pooled = x.mean(dim=1)
        return pooled, attn_weights


class SpatioTemporalDeepfakeCNN(nn.Module):
    def __init__(self, model_name='efficientnet_b4', hidden_dim=256, dropout=0.4, pretrained=True,
                 temporal_type='bilstm_attention', lstm_hidden=256, lstm_layers=2, attention_heads=4,
                 freeze_backbone=False, use_gradient_checkpointing=False):
        super().__init__()
        self.temporal_type = temporal_type
        # P100: NO gradient checkpointing
        self.use_gradient_checkpointing = False

        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.backbone_dim = self.backbone.num_features

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        if temporal_type in ['bilstm', 'bilstm_attention']:
            self.temporal = nn.LSTM(input_size=self.backbone_dim, hidden_size=lstm_hidden, num_layers=lstm_layers,
                                    batch_first=True, bidirectional=True, dropout=dropout if lstm_layers > 1 else 0)
            # P100 FIX: Disable flatten_parameters
            self.temporal.flatten_parameters = lambda: None
            temporal_out_dim = lstm_hidden * 2
            if temporal_type == 'bilstm_attention':
                self.temporal_attention = TemporalAttention(feature_dim=lstm_hidden * 2, num_heads=attention_heads, dropout=dropout)
        elif temporal_type == 'transformer':
            encoder_layer = nn.TransformerEncoderLayer(d_model=self.backbone_dim, nhead=attention_heads,
                                                       dim_feedforward=self.backbone_dim * 2, dropout=dropout, activation='gelu', batch_first=True)
            self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=lstm_layers)
            self.temporal_attention = TemporalAttention(feature_dim=self.backbone_dim, num_heads=attention_heads, dropout=dropout)
            temporal_out_dim = self.backbone_dim
        else:
            raise ValueError(f"Unknown temporal_type: {temporal_type}")

        self.temporal_out_dim = temporal_out_dim
        self.classifier = nn.Sequential(
            nn.Linear(temporal_out_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.GELU(), nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1)
        )
        self._init_weights()
        print(f"\n✓ SpatioTemporalDeepfakeCNN (P100 Compatible)")
        print(f"  Backbone: {model_name}, Temporal: {temporal_type}")
        print(f"  ⚠️ P100 MODE: No checkpointing, LSTM cuDNN bypassed")

    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_frame_features(self, frames):
        B, T, C, H, W = frames.shape
        frames_flat = frames.view(B * T, C, H, W)
        # P100: Direct forward, NO checkpointing
        features_flat = self.backbone(frames_flat)
        return features_flat.view(B, T, -1)

    def forward(self, frames, mask=None):
        features = self.extract_frame_features(frames)
        # P100: Wrap LSTM in cudnn.flags(enabled=False)
        if self.temporal_type == 'bilstm':
            with torch.backends.cudnn.flags(enabled=False):
                lstm_out, _ = self.temporal(features)
            if mask is not None:
                mask_expanded = mask.unsqueeze(-1).float()
                pooled = (lstm_out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
            else:
                pooled = lstm_out.mean(dim=1)
        elif self.temporal_type == 'bilstm_attention':
            with torch.backends.cudnn.flags(enabled=False):
                lstm_out, _ = self.temporal(features)
            pooled, _ = self.temporal_attention(lstm_out, mask)
        elif self.temporal_type == 'transformer':
            attn_mask = ~mask if mask is not None else None
            transformer_out = self.temporal(features, src_key_padding_mask=attn_mask)
            pooled, _ = self.temporal_attention(transformer_out, mask)
        return self.classifier(pooled).squeeze(-1)

    def get_attention_weights(self, frames, mask=None):
        with torch.no_grad():
            features = self.extract_frame_features(frames)
            if self.temporal_type == 'bilstm_attention':
                with torch.backends.cudnn.flags(enabled=False):
                    lstm_out, _ = self.temporal(features)
                _, attn_weights = self.temporal_attention(lstm_out, mask)
                return attn_weights
            elif self.temporal_type == 'transformer':
                transformer_out = self.temporal(features, src_key_padding_mask=~mask if mask is not None else None)
                _, attn_weights = self.temporal_attention(transformer_out, mask)
                return attn_weights
            return None


class DeepfakeCNN(nn.Module):
    def __init__(self, model_name='efficientnet_b4', hidden_dim=256, dropout=0.4, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.classifier = nn.Sequential(nn.Linear(self.backbone.num_features, hidden_dim), nn.BatchNorm1d(hidden_dim),
                                         nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)


print("\n" + "="*70)
print("INITIALIZING MODEL (P100 COMPATIBLE)")
print("="*70)

model = SpatioTemporalDeepfakeCNN(
    model_name=cfg.MODEL_NAME, hidden_dim=cfg.HIDDEN_DIM, dropout=cfg.DROPOUT, pretrained=True,
    temporal_type=cfg.TEMPORAL_TYPE, lstm_hidden=cfg.LSTM_HIDDEN, lstm_layers=cfg.LSTM_LAYERS,
    attention_heads=cfg.ATTENTION_HEADS, freeze_backbone=cfg.FREEZE_BACKBONE, use_gradient_checkpointing=False
)
model = model.to(DEVICE)
print(f"✓ Model on {DEVICE}")

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_params:,} total, {trainable:,} trainable")


## 5. Training Protocol

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES (P100 COMPATIBLE - STRICT FP32, NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════
# IMPROVEMENTS:
# 1. Optimal threshold finding for better F1
# 2. Reduced gradient clipping (0.5 instead of 1.0)
# 3. Better NaN handling
# ═══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """Focal Loss for class imbalance."""
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p = torch.sigmoid(inputs)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        focal_loss = alpha_t * focal_weight * bce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


def compute_eer(y_true: np.ndarray, y_scores: np.ndarray) -> float:
    """Compute Equal Error Rate."""
    try:
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        fnr = 1 - tpr
        eer = brentq(lambda x: interp1d(fpr, fnr)(x) - x, 0.0, 1.0)
        return float(eer)
    except:
        return 0.5


def find_optimal_threshold_cnn(y_true, y_prob):
    """Find threshold that maximizes F1 score."""
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.2, 0.8, 0.02):
        preds = (y_prob >= thresh).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, min_lr=1e-7):
    """Cosine schedule with linear warmup."""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(min_lr / optimizer.defaults['lr'], 0.5 * (1.0 + np.cos(np.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch_with_accumulation(model, loader, criterion, optimizer, scheduler, device, accumulation_steps=2):
    """Train one epoch - P100 COMPATIBLE (NO AMP)."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    num_samples = 0
    current_lr = optimizer.param_groups[0]['lr']
    nan_batches = 0

    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, batch in enumerate(pbar):
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        mask = batch['mask'].to(device)

        # P100: STRICT FP32 - NO autocast
        logits = model(frames, mask)
        
        # NaN check
        if torch.isnan(logits).any():
            nan_batches += 1
            continue
        
        loss = criterion(logits, labels) / accumulation_steps
        
        if torch.isnan(loss):
            nan_batches += 1
            continue

        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            # Reduced gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']

        batch_size = labels.size(0)
        total_loss += (loss.item() * accumulation_steps) * batch_size
        num_samples += batch_size
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item() * accumulation_steps:.4f}', 'lr': f'{current_lr:.2e}'})

    # Handle remaining gradients
    if (batch_idx + 1) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        optimizer.zero_grad()

    if nan_batches > 0:
        print(f"  ⚠️ {nan_batches} NaN batches skipped")

    avg_loss = total_loss / max(num_samples, 1)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Use optimal threshold for training metrics too
    opt_thresh, _ = find_optimal_threshold_cnn(all_labels, all_preds)
    y_pred_binary = (all_preds >= opt_thresh).astype(int)
    
    acc = accuracy_score(all_labels, y_pred_binary)
    if len(np.unique(all_labels)) > 1:
        auc = roc_auc_score(all_labels, all_preds)
        eer = compute_eer(all_labels, all_preds)
    else:
        auc = 0.5
        eer = 0.5
    
    return avg_loss, acc, auc, eer, current_lr


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate with optimal threshold finding."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    num_samples = 0

    for batch in tqdm(loader, desc="Validating", leave=False):
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        mask = batch['mask'].to(device)

        # P100: STRICT FP32 - NO autocast
        logits = model(frames, mask)
        
        if torch.isnan(logits).any():
            continue
            
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        num_samples += batch_size
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(num_samples, 1)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # IMPROVEMENT: Find optimal threshold
    opt_thresh, opt_f1 = find_optimal_threshold_cnn(all_labels, all_preds)
    y_pred_binary = (all_preds >= opt_thresh).astype(int)
    
    acc = accuracy_score(all_labels, y_pred_binary)
    if len(np.unique(all_labels)) > 1:
        auc = roc_auc_score(all_labels, all_preds)
        eer = compute_eer(all_labels, all_preds)
    else:
        auc = 0.5
        eer = 0.5
    
    f1 = f1_score(all_labels, y_pred_binary, zero_division=0)
    precision = precision_score(all_labels, y_pred_binary, zero_division=0)
    recall = recall_score(all_labels, y_pred_binary, zero_division=0)
    
    # Check prediction distribution
    n_pred_0 = (y_pred_binary == 0).sum()
    n_pred_1 = (y_pred_binary == 1).sum()
    if n_pred_0 == 0 or n_pred_1 == 0:
        print(f"  ⚠️ Single class prediction! Preds: 0={n_pred_0}, 1={n_pred_1}")
    
    return {
        'loss': avg_loss, 
        'acc': acc, 
        'auc': auc, 
        'eer': eer, 
        'f1': f1, 
        'precision': precision, 
        'recall': recall,
        'optimal_threshold': opt_thresh,
        'preds': all_preds, 
        'labels': all_labels
    }


print("✓ Training utilities loaded (P100 - FP32 only)")
print("  - Optimal threshold finding: ENABLED")
print("  - Gradient clipping: 0.5 (reduced)")
print("  - NaN handling: ENABLED")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP (P100 COMPATIBLE - STRICT FP32, NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════

criterion = FocalLoss(alpha=cfg.FOCAL_ALPHA, gamma=cfg.FOCAL_GAMMA)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)

steps_per_epoch = len(train_loader) // cfg.GRAD_ACCUMULATION_STEPS
total_training_steps = cfg.NUM_EPOCHS * steps_per_epoch
warmup_steps = int(cfg.WARMUP_RATIO * total_training_steps)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

# P100: NO GradScaler

history = {'train_loss': [], 'train_acc': [], 'train_auc': [], 'train_eer': [],
           'val_loss': [], 'val_acc': [], 'val_auc': [], 'val_eer': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'lr': []}

best_val_auc = 0.0
best_val_eer = 1.0
best_epoch = 0
patience = 5
patience_counter = 0

print("\n" + "="*70)
print("TRAINING (P100 COMPATIBLE - STRICT FP32)")
print("="*70)
print(f"  ⚠️ Mixed Precision: DISABLED (P100 lacks Tensor Cores)")
print(f"  Physical batch: {cfg.BATCH_SIZE}, Accumulation: {cfg.GRAD_ACCUMULATION_STEPS}, Effective: {cfg.BATCH_SIZE * cfg.GRAD_ACCUMULATION_STEPS}")
print(f"  Epochs: {cfg.NUM_EPOCHS}, LR: {cfg.LEARNING_RATE}, Warmup: {warmup_steps}")
print(f"  Loss: Focal Loss (α={cfg.FOCAL_ALPHA}, γ={cfg.FOCAL_GAMMA})")
if cfg.CURRENT_FOLD >= 0:
    print(f"  K-Fold: Fold {cfg.CURRENT_FOLD + 1}/{cfg.K_FOLDS}")
if cfg.USE_IDENTITY_SPLIT:
    print(f"  Identity-based split: Enabled")
print("="*70 + "\n")

for epoch in range(cfg.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS}")
    print("-" * 50)

    train_loss, train_acc, train_auc, train_eer, current_lr = train_one_epoch_with_accumulation(
        model=model, loader=train_loader, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler, device=DEVICE, accumulation_steps=cfg.GRAD_ACCUMULATION_STEPS
    )

    val_metrics = validate(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_eer = val_metrics['loss'], val_metrics['acc'], val_metrics['auc'], val_metrics['eer']
    val_f1, val_precision, val_recall = val_metrics['f1'], val_metrics['precision'], val_metrics['recall']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['train_eer'].append(train_eer)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_eer'].append(val_eer)
    history['val_f1'].append(val_f1)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['lr'].append(current_lr)

    print(f"  Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f} | EER: {train_eer:.4f}")
    print(f"  Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | EER: {val_eer:.4f}")
    print(f"        | F1: {val_f1:.4f} | Prec: {val_precision:.4f} | Rec: {val_recall:.4f} | LR: {current_lr:.2e}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_eer = val_eer
        best_epoch = epoch + 1
        patience_counter = 0
        model_name = f"best_cnn_model_fold{cfg.CURRENT_FOLD}.pth" if cfg.CURRENT_FOLD >= 0 else "best_cnn_model.pth"
        torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, model_name))
        print(f"  ★ New best! (AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n  ⚠ Early stopping at epoch {epoch+1}")
            break

print("\n" + "="*70)
print("TRAINING COMPLETE")
print(f"  Best Epoch: {best_epoch}, Best AUC: {best_val_auc:.4f}, Best EER: {best_val_eer:.4f}")
print("="*70)

history_path = os.path.join(cfg.OUTPUT_DIR, f"training_history_fold{cfg.CURRENT_FOLD}.json" if cfg.CURRENT_FOLD >= 0 else "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"History saved to: {history_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING CURVES (Research-Grade Visualization)
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = range(1, len(history['train_loss']) + 1)

# 1. Loss Curve
axes[0, 0].plot(epochs, history['train_loss'], 'b-', marker='o', label='Train', linewidth=2, markersize=4)
axes[0, 0].plot(epochs, history['val_loss'], 'r-', marker='s', label='Val', linewidth=2, markersize=4)
axes[0, 0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best (Epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Loss Curve', fontsize=12, fontweight='bold')
axes[0, 0].legend(loc='upper right')
axes[0, 0].grid(True, alpha=0.3)

# 2. AUC Curve
axes[0, 1].plot(epochs, history['train_auc'], 'b-', marker='o', label='Train', linewidth=2, markersize=4)
axes[0, 1].plot(epochs, history['val_auc'], 'r-', marker='s', label='Val', linewidth=2, markersize=4)
axes[0, 1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best (Epoch {best_epoch})')
axes[0, 1].axhline(y=best_val_auc, color='green', linestyle=':', alpha=0.5)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('AUC-ROC', fontsize=11)
axes[0, 1].set_title(f'AUC Curve (Best: {best_val_auc:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].legend(loc='lower right')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0.4, 1.0])

# 3. F1 / Precision / Recall
axes[1, 0].plot(epochs, history['val_f1'], 'g-', marker='o', label='F1', linewidth=2, markersize=4)
axes[1, 0].plot(epochs, history['val_precision'], 'b-', marker='^', label='Precision', linewidth=2, markersize=4)
axes[1, 0].plot(epochs, history['val_recall'], 'r-', marker='v', label='Recall', linewidth=2, markersize=4)
axes[1, 0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch')
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Score', fontsize=11)
axes[1, 0].set_title('Validation Metrics (F1, Precision, Recall)', fontsize=12, fontweight='bold')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0.0, 1.0])

# 4. Learning Rate Schedule
axes[1, 1].plot(epochs, history['lr'], 'purple', marker='o', linewidth=2, markersize=4)
axes[1, 1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch')
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Learning Rate', fontsize=11)
axes[1, 1].set_title('Learning Rate Schedule (Warmup + Cosine)', fontsize=12, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(loc='upper right')

# Add warmup region annotation
if len(history['lr']) > 1:
    warmup_end_epoch = int(cfg.WARMUP_RATIO * cfg.NUM_EPOCHS) + 1
    if warmup_end_epoch > 0 and warmup_end_epoch < len(epochs):
        axes[1, 1].axvspan(1, warmup_end_epoch, alpha=0.2, color='yellow', label='Warmup Phase')

plt.suptitle(f'CNN Spatio-Temporal Training Progress\n{cfg.EXPERIMENT_NAME} {cfg.EXPERIMENT_VERSION}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

# Save figure
fig_path = os.path.join(cfg.OUTPUT_DIR, f'training_curves_fold{cfg.CURRENT_FOLD}.png' if cfg.CURRENT_FOLD >= 0 else 'training_curves.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f"✓ Training curves saved to: {fig_path}")

plt.show()


## 6. Video-Level Inference & Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD BEST MODEL
# ═══════════════════════════════════════════════════════════════════════════════

# Load best model weights
model.load_state_dict(torch.load(os.path.join(cfg.OUTPUT_DIR, "best_cnn_model.pth"), map_location=DEVICE, weights_only=True))
model.eval()
print("✓ Best model loaded for inference")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO-LEVEL PREDICTION (P100 COMPATIBLE - NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════

def predict_video_temporal(model, faces: List[np.ndarray], transform, device, max_frames=15):
    """Predict deepfake probability for a video using temporal model."""
    model.eval()
    
    if len(faces) < 3:
        return 0.5
    
    indices = np.linspace(0, len(faces) - 1, max_frames, dtype=int) if len(faces) >= max_frames else list(range(len(faces)))
    selected_faces = [faces[i] for i in indices]
    
    while len(selected_faces) < max_frames:
        selected_faces.append(faces[-1])
    
    frame_tensors = [transform(image=face)['image'] for face in selected_faces]
    frames = torch.stack(frame_tensors).unsqueeze(0).to(device)
    mask = torch.ones(1, max_frames, dtype=torch.bool).to(device)
    mask[:, len(faces):] = False
    
    with torch.no_grad():
        # P100: NO autocast - STRICT FP32
        logit = model(frames, mask)
        prob = torch.sigmoid(logit).item()
    
    return prob


def generate_video_predictions(model, all_videos, face_data, transform, device):
    """Generate predictions for all videos."""
    predictions = []
    
    for video in tqdm(all_videos, desc="Predicting"):
        video_id = video['video_id']
        if video_id in face_data:
            prob = predict_video_temporal(model, face_data[video_id], transform, device)
        else:
            prob = 0.5
        predictions.append({
            'video_id': video_id,
            'label': video['label'],
            'prediction': prob,
            'predicted_class': 1 if prob > 0.5 else 0
        })
    
    return predictions


print("✓ Video prediction functions loaded (P100 compatible)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT PREDICTIONS WITH BOOTSTRAP CONFIDENCE INTERVALS (Research Requirement)
# ═══════════════════════════════════════════════════════════════════════════════

def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstrap=1000, ci=95, seed=42):
    """
    Compute bootstrap confidence intervals for a metric.
    
    RESEARCH REQUIREMENT: Papers must report confidence intervals to demonstrate
    statistical significance and reliability of reported metrics.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predictions (probabilities for AUC, binary for accuracy/F1)
        metric_fn: Function(y_true, y_pred) -> float
        n_bootstrap: Number of bootstrap iterations
        ci: Confidence level (95 = 95% CI)
        seed: Random seed for reproducibility
    
    Returns:
        (lower_bound, point_estimate, upper_bound)
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    
    for _ in range(n_bootstrap):
        indices = rng.randint(0, n, n)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        # Skip if only one class present
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            score = metric_fn(y_true_boot, y_pred_boot)
            scores.append(score)
        except Exception:
            continue
    
    if len(scores) == 0:
        point = metric_fn(y_true, y_pred)
        return point, point, point
    
    scores = np.array(scores)
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    point = metric_fn(y_true, y_pred)
    
    return lower, point, upper


# Add ground truth labels
video_labels = {v['video_id']: v['label'] for v in all_videos}
predictions_df['true_label'] = predictions_df['video_id'].map(video_labels)

# Save predictions CSV
csv_path = os.path.join(cfg.OUTPUT_DIR, f"cnn_predictions_fold{cfg.CURRENT_FOLD}.csv" if cfg.CURRENT_FOLD >= 0 else "cnn_predictions.csv")
predictions_df.to_csv(csv_path, index=False)
print(f"\n✓ Predictions saved to: {csv_path}")

# Alias for metrics below
predictions_df['label'] = predictions_df['true_label']

# Calculate video-level metrics
y_true = predictions_df['label'].values
y_pred_proba = predictions_df['P_CNN'].values
y_pred = (y_pred_proba > 0.5).astype(int)

# ═══════════════════════════════════════════════════════════════════════════════
# METRICS WITH 95% CONFIDENCE INTERVALS (RESEARCH REQUIREMENT)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("VIDEO-LEVEL METRICS WITH 95% CONFIDENCE INTERVALS")
print("="*70)

# AUC with CI
if len(np.unique(y_true)) > 1:
    auc_lo, auc_pt, auc_hi = bootstrap_ci(y_true, y_pred_proba, roc_auc_score)
    print(f"  AUC-ROC:    {auc_pt:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")
else:
    auc_pt = 0.5
    print(f"  AUC-ROC:    {auc_pt:.4f} (only one class in predictions)")

# Accuracy with CI
acc_lo, acc_pt, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_score)
print(f"  Accuracy:   {acc_pt:.4f} [{acc_lo:.4f}, {acc_hi:.4f}]")

# F1 with CI
f1_lo, f1_pt, f1_hi = bootstrap_ci(y_true, y_pred, f1_score)
print(f"  F1-Score:   {f1_pt:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]")

# Precision with CI
from sklearn.metrics import precision_score, recall_score
prec_lo, prec_pt, prec_hi = bootstrap_ci(y_true, y_pred, lambda yt, yp: precision_score(yt, yp, zero_division=0))
print(f"  Precision:  {prec_pt:.4f} [{prec_lo:.4f}, {prec_hi:.4f}]")

# Recall with CI
rec_lo, rec_pt, rec_hi = bootstrap_ci(y_true, y_pred, lambda yt, yp: recall_score(yt, yp, zero_division=0))
print(f"  Recall:     {rec_pt:.4f} [{rec_lo:.4f}, {rec_hi:.4f}]")

print("="*70)

# Store metrics for later use
video_metrics = {
    'auc': {'point': auc_pt, 'ci_lower': auc_lo if 'auc_lo' in dir() else auc_pt, 'ci_upper': auc_hi if 'auc_hi' in dir() else auc_pt},
    'accuracy': {'point': acc_pt, 'ci_lower': acc_lo, 'ci_upper': acc_hi},
    'f1': {'point': f1_pt, 'ci_lower': f1_lo, 'ci_upper': f1_hi},
    'precision': {'point': prec_pt, 'ci_lower': prec_lo, 'ci_upper': prec_hi},
    'recall': {'point': rec_pt, 'ci_lower': rec_lo, 'ci_upper': rec_hi},
}

# Save metrics JSON
metrics_path = os.path.join(cfg.OUTPUT_DIR, f"metrics_fold{cfg.CURRENT_FOLD}.json" if cfg.CURRENT_FOLD >= 0 else "metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(video_metrics, f, indent=2)
print(f"\n✓ Metrics with CI saved to: {metrics_path}")

# Print distribution info
print("\nP_CNN Distribution:")
print(predictions_df['P_CNN'].describe())


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RESEARCH-GRADE EVALUATION VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════════════════════
#
# This cell generates all visualizations required for research papers:
#   1. ROC Curve with AUC
#   2. Precision-Recall Curve with AP
#   3. Confusion Matrix
#   4. Score Distribution by Class
#   5. Classification Report
#
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, precision_recall_curve, average_precision_score,
    roc_curve, classification_report
)
import seaborn as sns

# Prepare data
real_preds = predictions_df[predictions_df['label'] == 0]['P_CNN']
fake_preds = predictions_df[predictions_df['label'] == 1]['P_CNN']

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ═══════════════════════════════════════════════════════════════════════════════
# 1. ROC Curve
# ═══════════════════════════════════════════════════════════════════════════════
fpr, tpr, thresholds_roc = roc_curve(y_true, y_pred_proba)
roc_auc = roc_auc_score(y_true, y_pred_proba) if len(np.unique(y_true)) > 1 else 0.5

axes[0, 0].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'CNN (AUC = {roc_auc:.4f})')
axes[0, 0].fill_between(fpr, tpr, alpha=0.2, color='blue')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1.5, label='Random')
axes[0, 0].set_xlabel('False Positive Rate', fontsize=11)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=11)
axes[0, 0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0, 0].legend(loc='lower right', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim([-0.02, 1.02])
axes[0, 0].set_ylim([-0.02, 1.02])

# Find optimal threshold (Youden's J)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds_roc[optimal_idx]
axes[0, 0].scatter(fpr[optimal_idx], tpr[optimal_idx], marker='*', s=200, c='red', 
                   zorder=5, label=f'Optimal (t={optimal_threshold:.2f})')

# ═══════════════════════════════════════════════════════════════════════════════
# 2. Precision-Recall Curve (REQUIRED for imbalanced datasets)
# ═══════════════════════════════════════════════════════════════════════════════
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_true, y_pred_proba)
ap = average_precision_score(y_true, y_pred_proba)

# Baseline (random classifier)
baseline = y_true.sum() / len(y_true)

axes[0, 1].plot(recall_curve, precision_curve, 'g-', linewidth=2.5, label=f'CNN (AP = {ap:.4f})')
axes[0, 1].fill_between(recall_curve, precision_curve, alpha=0.2, color='green')
axes[0, 1].axhline(y=baseline, color='k', linestyle='--', alpha=0.5, linewidth=1.5, label=f'Baseline ({baseline:.2f})')
axes[0, 1].set_xlabel('Recall', fontsize=11)
axes[0, 1].set_ylabel('Precision', fontsize=11)
axes[0, 1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[0, 1].legend(loc='lower left', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim([-0.02, 1.02])
axes[0, 1].set_ylim([-0.02, 1.02])

# ═══════════════════════════════════════════════════════════════════════════════
# 3. Confusion Matrix (REQUIRED for all papers)
# ═══════════════════════════════════════════════════════════════════════════════
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Create annotation with both count and percentage
annot = np.array([[f'{cm[i,j]}\n({cm_normalized[i,j]*100:.1f}%)' 
                   for j in range(2)] for i in range(2)])

sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'}, annot_kws={'fontsize': 14})
axes[1, 0].set_xlabel('Predicted Label', fontsize=11)
axes[1, 0].set_ylabel('True Label', fontsize=11)
axes[1, 0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# ═══════════════════════════════════════════════════════════════════════════════
# 4. Score Distribution by Class
# ═══════════════════════════════════════════════════════════════════════════════
bins = np.linspace(0, 1, 31)

axes[1, 1].hist(real_preds, bins=bins, alpha=0.6, label=f'Real (n={len(real_preds)})', 
                color='green', density=True, edgecolor='darkgreen', linewidth=1)
axes[1, 1].hist(fake_preds, bins=bins, alpha=0.6, label=f'Fake (n={len(fake_preds)})', 
                color='red', density=True, edgecolor='darkred', linewidth=1)
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1, 1].axvline(x=optimal_threshold, color='purple', linestyle=':', linewidth=2, 
                   label=f'Optimal={optimal_threshold:.2f}')
axes[1, 1].set_xlabel('P(Fake)', fontsize=11)
axes[1, 1].set_ylabel('Density', fontsize=11)
axes[1, 1].set_title('Score Distribution by Class', fontsize=12, fontweight='bold')
axes[1, 1].legend(loc='upper center', fontsize=9)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim([-0.02, 1.02])

plt.suptitle(f'CNN Spatio-Temporal Evaluation Results\n{cfg.EXPERIMENT_NAME}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

# Save figure
fig_path = os.path.join(cfg.OUTPUT_DIR, f'evaluation_metrics_fold{cfg.CURRENT_FOLD}.png' if cfg.CURRENT_FOLD >= 0 else 'evaluation_metrics.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f"\n✓ Evaluation visualizations saved to: {fig_path}")

plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATION REPORT (REQUIRED for all papers)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred, target_names=['Real', 'Fake'], digits=4))

# Print additional statistics
print("\n" + "="*70)
print("ADDITIONAL STATISTICS")
print("="*70)
print(f"  Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  Average Precision (AP): {ap:.4f}")
print(f"  Class balance: Real={len(real_preds)}, Fake={len(fake_preds)} ({len(fake_preds)/len(y_true)*100:.1f}% fake)")
print(f"  Mean P_CNN (Real): {real_preds.mean():.4f} ± {real_preds.std():.4f}")
print(f"  Mean P_CNN (Fake): {fake_preds.mean():.4f} ± {fake_preds.std():.4f}")
print("="*70)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRAD-CAM VISUALIZATION (P100 COMPATIBLE)
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalGradCAM:
    def __init__(self, model, target_layer=None):
        self.model = model
        self.gradients = None
        self.activations = None
        self.handles = []
        if target_layer is None:
            self.target_layer = model.backbone.conv_head
        else:
            self.target_layer = target_layer
        self.handles.append(self.target_layer.register_forward_hook(self._save_activation))
        self.handles.append(self.target_layer.register_full_backward_hook(self._save_gradient))

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def generate(self, frames: torch.Tensor, mask: torch.Tensor = None, target_frame_idx: int = 0, class_idx: int = 1):
        self.model.eval()
        B, T, C, H, W = frames.shape
        assert B == 1 and 0 <= target_frame_idx < T

        frames.requires_grad_(True)

        # P100: NO autocast - STRICT FP32
        logit = self.model(frames, mask)
        prob = torch.sigmoid(logit).item()

        self.model.zero_grad()
        if class_idx == 1:
            logit.backward(retain_graph=True)
        else:
            (-logit).backward(retain_graph=True)

        if self.gradients is None or self.activations is None:
            return np.zeros((H, W)), prob

        target_gradients = self.gradients[target_frame_idx:target_frame_idx+1]
        target_activations = self.activations[target_frame_idx:target_frame_idx+1]

        weights = target_gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * target_activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = torch.zeros_like(cam)

        cam = F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy(), prob

    def generate_multi_frame(self, frames, mask=None, class_idx=1):
        B, T, C, H, W = frames.shape
        heatmaps = []
        prob = None
        for frame_idx in range(T):
            cam, prob = self.generate(frames.clone(), mask, target_frame_idx=frame_idx, class_idx=class_idx)
            heatmaps.append(cam)
        return heatmaps, prob


def visualize_gradcam(model, video_data, face_data, val_transforms, device, num_videos=3):
    gradcam = TemporalGradCAM(model)
    fig, axes = plt.subplots(num_videos, 5, figsize=(15, 3*num_videos))
    
    videos_shown = 0
    for video in video_data[:20]:
        if videos_shown >= num_videos:
            break
        video_id = video['video_id']
        if video_id not in face_data or len(face_data[video_id]) < 5:
            continue

        faces = face_data[video_id]
        indices = np.linspace(0, len(faces) - 1, 15, dtype=int)
        selected_faces = [faces[i] for i in indices]

        frame_tensors = [val_transforms(image=face)['image'] for face in selected_faces]
        frames = torch.stack(frame_tensors).unsqueeze(0).to(device)
        mask = torch.ones(1, 15, dtype=torch.bool).to(device)

        for col, frame_idx in enumerate([0, 3, 7, 11, 14]):
            cam, prob = gradcam.generate(frames.clone(), mask, target_frame_idx=frame_idx)
            ax = axes[videos_shown, col] if num_videos > 1 else axes[col]
            ax.imshow(selected_faces[frame_idx])
            ax.imshow(cam, cmap='jet', alpha=0.5)
            ax.set_title(f"Frame {frame_idx}\nP(fake)={prob:.2f}")
            ax.axis('off')
        videos_shown += 1

    gradcam.remove_hooks()
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Grad-CAM saved")

print("✓ Grad-CAM loaded (P100 compatible)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ABLATION STUDY FRAMEWORK (P100 COMPATIBLE - NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════

def run_single_ablation(config, train_videos, val_videos, face_data, device, epochs=5):
    """Run a single ablation experiment - P100 COMPATIBLE."""
    
    train_dataset = DeepfakeVideoDataset(train_videos, face_data, transform=get_train_transforms(), max_frames=cfg.FRAMES_PER_VIDEO)
    val_dataset = DeepfakeVideoDataset(val_videos, face_data, transform=get_val_transforms(), max_frames=cfg.FRAMES_PER_VIDEO)
    
    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    
    model = SpatioTemporalDeepfakeCNN(
        model_name=config.get('model_name', cfg.MODEL_NAME),
        hidden_dim=config.get('hidden_dim', cfg.HIDDEN_DIM),
        dropout=config.get('dropout', cfg.DROPOUT),
        temporal_type=config.get('temporal_type', cfg.TEMPORAL_TYPE),
        lstm_hidden=config.get('lstm_hidden', cfg.LSTM_HIDDEN),
        lstm_layers=config.get('lstm_layers', cfg.LSTM_LAYERS),
        attention_heads=config.get('attention_heads', cfg.ATTENTION_HEADS),
        use_gradient_checkpointing=False  # P100: Always False
    ).to(device)
    
    criterion = FocalLoss(alpha=cfg.FOCAL_ALPHA, gamma=cfg.FOCAL_GAMMA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    
    # P100: NO GradScaler
    
    best_auc = 0.0
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            frames = batch['frames'].to(device)
            labels = batch['label'].to(device)
            mask = batch['mask'].to(device)
            
            optimizer.zero_grad()
            # P100: NO autocast - STRICT FP32
            logits = model(frames, mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        val_metrics = validate(model, val_loader, criterion, device)
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_auc


def run_ablation_study():
    """Run ablation study comparing different configurations."""
    print("\n" + "="*70)
    print("ABLATION STUDY (P100 COMPATIBLE)")
    print("="*70)
    
    ablations = [
        {'name': 'BiLSTM (no attention)', 'temporal_type': 'bilstm'},
        {'name': 'BiLSTM + Attention', 'temporal_type': 'bilstm_attention'},
    ]
    
    results = []
    for ablation in ablations:
        print(f"\nRunning: {ablation['name']}")
        config = {k: v for k, v in ablation.items() if k != 'name'}
        auc = run_single_ablation(config, train_videos, val_videos, face_data, DEVICE, epochs=3)
        results.append({'name': ablation['name'], 'auc': auc})
        print(f"  AUC: {auc:.4f}")
    
    print("\n" + "="*70)
    print("ABLATION RESULTS")
    print("="*70)
    for r in sorted(results, key=lambda x: x['auc'], reverse=True):
        print(f"  {r['name']}: AUC={r['auc']:.4f}")
    
    return results


print("✓ Ablation study loaded (P100 compatible)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

# Save final model weights
torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "cnn_spatial_stream_final.pth"))

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_training_history.csv"), index=False)

# Save config
config_dict = {
    'model_name': cfg.MODEL_NAME,
    'hidden_dim': cfg.HIDDEN_DIM,
    'dropout': cfg.DROPOUT,
    'img_size': cfg.IMG_SIZE,
    'frames_per_video': cfg.FRAMES_PER_VIDEO,
    'batch_size': cfg.BATCH_SIZE,
    'learning_rate': cfg.LEARNING_RATE,
    'num_epochs': cfg.NUM_EPOCHS,
    'best_val_auc': best_val_auc,
    'video_level_auc': video_auc,
}
pd.DataFrame([config_dict]).to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_config.csv"), index=False)

print("\n" + "="*70)
print("ALL ARTIFACTS SAVED")
print("="*70)
print(f"  ✓ cnn_predictions.csv        (video-level P_CNN scores)")
print(f"  ✓ best_cnn_model.pth          (best model weights)")
print(f"  ✓ cnn_spatial_stream_final.pth (final model weights)")
print(f"  ✓ cnn_training_history.csv   (training metrics)")
print(f"  ✓ cnn_config.csv             (configuration)")
print(f"  ✓ training_curves.png        (loss/acc/auc plots)")
print(f"  ✓ cnn_results.png            (prediction distribution)")
print("="*70)

## 7. Late Fusion Integration Guide

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LATE FUSION INTEGRATION GUIDE
# ═══════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                       LATE FUSION INTEGRATION GUIDE                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  This notebook outputs: cnn_predictions.csv                                  ║
║  Columns: video_id, P_CNN                                                    ║
║                                                                              ║
║  To fuse with rPPG predictions from 2ND_MODEL.ipynb:                        ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │  # Load both predictions                                                │ ║
║  │  cnn_df = pd.read_csv('cnn_predictions.csv')                           │ ║
║  │  rppg_df = pd.read_csv('rppg_predictions.csv')                         │ ║
║  │                                                                         │ ║
║  │  # Merge on video_id                                                    │ ║
║  │  fused_df = cnn_df.merge(rppg_df, on='video_id')                       │ ║
║  │                                                                         │ ║
║  │  # Simple average fusion                                                │ ║
║  │  fused_df['P_final'] = (fused_df['P_CNN'] + fused_df['P_rPPG']) / 2    │ ║
║  │                                                                         │ ║
║  │  # Weighted fusion (if CNN is more accurate)                           │ ║
║  │  w_cnn, w_rppg = 0.6, 0.4                                              │ ║
║  │  fused_df['P_final'] = w_cnn * fused_df['P_CNN']                       │ ║
║  │                      + w_rppg * fused_df['P_rPPG']                      │ ║
║  │                                                                         │ ║
║  │  # Learned fusion (train a small classifier)                           │ ║
║  │  from sklearn.linear_model import LogisticRegression                   │ ║
║  │  X_fusion = fused_df[['P_CNN', 'P_rPPG']].values                       │ ║
║  │  y_fusion = fused_df['label'].values                                   │ ║
║  │  fusion_model = LogisticRegression().fit(X_fusion, y_fusion)           │ ║
║  │  fused_df['P_final'] = fusion_model.predict_proba(X_fusion)[:, 1]      │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("CNN SPATIO-TEMPORAL STREAM — FINAL SUMMARY")
print("="*70)

print(f"""
  ┌─────────────────────────────────────────────────────────────────────┐
  │  ARCHITECTURE                                                        │
  ├─────────────────────────────────────────────────────────────────────┤
  │  Backbone:           EfficientNet-B4 (pretrained)                   │
  │  Temporal Model:     BiLSTM + Multi-Head Self-Attention             │
  │  LSTM Hidden:        {cfg.LSTM_HIDDEN} × 2 (bidirectional)                           │
  │  LSTM Layers:        {cfg.LSTM_LAYERS}                                                │
  │  Attention Heads:    {cfg.ATTENTION_HEADS}                                                │
  │  Total Parameters:   {total_params:,}                                  │
  │  Trainable Params:   {trainable_params:,}                                  │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  TRAINING                                                           │
  ├─────────────────────────────────────────────────────────────────────┤
  │  Training Videos:    {len(train_dataset)}                                              │
  │  Validation Videos:  {len(val_dataset)}                                              │
  │  Frames per Video:   {cfg.FRAMES_PER_VIDEO}                                               │
  │  Best Epoch:         {best_epoch}                                                │
  │  Best Val AUC:       {best_val_auc:.4f}                                           │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  VIDEO-LEVEL METRICS                                                │
  ├─────────────────────────────────────────────────────────────────────┤
  │  AUC-ROC:            {video_auc:.4f}                                           │
  │  Accuracy:           {video_acc:.4f}                                           │
  │  F1-Score:           {video_f1:.4f}                                           │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  OUTPUT FILES                                                       │
  ├─────────────────────────────────────────────────────────────────────┤
  │  ✓ cnn_predictions.csv         (video-level P_CNN scores)          │
  │  ✓ best_cnn_model.pth          (best model weights)                │
  │  ✓ gradcam_gallery.png         (model interpretability)            │
  │  ✓ training_curves.png         (loss/acc/auc plots)                │
  └─────────────────────────────────────────────────────────────────────┘
""")

print("="*70)
print("\n✅ SPATIO-TEMPORAL CNN TRAINING COMPLETE!")
print("   ✓ Temporal modeling via BiLSTM + Attention (detects inter-frame artifacts)")
print("   ✓ Grad-CAM visualization (proves model learns meaningful features)")
print("   ✓ Ready for Late Fusion with rPPG physiological stream")
print("="*70)
